In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# 1. Setup Input and Output Folders
input_folder = '../images/task1'
output_folder = '../results/task1'

# Automatically create the results folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Helper function to plot and SAVE the channels
def save_channels_plot(title, img_rgb, ch1_title, ch1, ch2_title, ch2, ch3_title, ch3, save_path):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(title, fontsize=16)
    
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original RGB")
    axes[0].axis('off')
    
    axes[1].imshow(ch1, cmap='gray')
    axes[1].set_title(ch1_title)
    axes[1].axis('off')
    
    axes[2].imshow(ch2, cmap='gray')
    axes[2].set_title(ch2_title)
    axes[2].axis('off')
    
    axes[3].imshow(ch3, cmap='gray')
    axes[3].set_title(ch3_title)
    axes[3].axis('off')
    
    # Save the figure to the results folder instead of showing it inline
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

# 2. Find all images in the task1 folder
image_paths = glob.glob(os.path.join(input_folder, '*.*'))

print(f"Found {len(image_paths)} images. Processing...")

# 3. Loop through every image found
for img_path in image_paths:
    # Extract just the file name (e.g., 'peppers.png') and its base name ('peppers')
    filename = os.path.basename(img_path)
    base_name, _ = os.path.splitext(filename)
    
    img_bgr = cv2.imread(img_path)
    
    # Skip if OpenCV couldn't read the file (e.g., if it's a hidden system file)
    if img_bgr is None:
        continue
        
    print(f"Processing {filename}...")
    
    # Convert to RGB for displaying
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # --- 1.1 RGB to HSV ---
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(img_hsv)
    save_channels_plot(f"HSV - {filename}", img_rgb, "Hue (H)", H, "Saturation (S)", S, "Value (V)", V, 
                       os.path.join(output_folder, f"{base_name}_1_HSV.png"))

    # --- 1.2 RGB to YCbCr ---
    img_ycbcr = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    Y, Cr, Cb = cv2.split(img_ycbcr)
    save_channels_plot(f"YCbCr - {filename}", img_rgb, "Luminance (Y)", Y, "Chrominance Red (Cr)", Cr, "Chrominance Blue (Cb)", Cb,
                       os.path.join(output_folder, f"{base_name}_2_YCbCr.png"))

    # --- 1.3 RGB to L*a*b* ---
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L, a, b = cv2.split(img_lab)
    save_channels_plot(f"L*a*b* - {filename}", img_rgb, "Lightness (L)", L, "Green-Red (a)", a, "Blue-Yellow (b)", b,
                       os.path.join(output_folder, f"{base_name}_3_LAB.png"))

    # --- 1.4 Histograms (RGB vs LAB) ---
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f"Color Histograms: RGB vs L*a*b* - {filename}", fontsize=16)

    # RGB Histogram
    colors = ('r', 'g', 'b')
    for i, col in enumerate(colors):
        hist = cv2.calcHist([img_rgb], [i], None, [256], [0, 256])
        axes[0].plot(hist, color=col)
        axes[0].set_title('RGB Histogram')
        axes[0].set_xlim([0, 256])

    # L*a*b* Histogram
    colors_lab = ('k', 'g', 'b') # Black for L, Green for a, Blue for b
    labels_lab = ('L (Lightness)', 'a (Green-Red)', 'b (Blue-Yellow)')
    for i, col in enumerate(colors_lab):
        hist = cv2.calcHist([img_lab], [i], None, [256], [0, 256])
        axes[1].plot(hist, color=col, label=labels_lab[i])
        axes[1].set_title('L*a*b* Histogram')
        axes[1].set_xlim([0, 256])
        axes[1].legend()

    # Save the histogram
    plt.savefig(os.path.join(output_folder, f"{base_name}_4_Histograms.png"), bbox_inches='tight')
    plt.close(fig)

print(f"Done! Check the '{output_folder}' folder for your results.")

Found 9 images. Processing...
Processing testRGB.bmp...
Processing lighthouse.png...
Processing floresVermelhas.bmp...
Processing folhasVerdes.bmp...
Processing yellowlily.jpg...
Processing strawberries.jpg...
Processing lena.bmp...
Processing nenufares.bmp...
Processing peppers.png...
Done! Check the '../results/task1' folder for your results.


### Task 1 Analysis: Experiments with Color Spaces

#### 1.1 RGB to HSV Conversion
* **Observations (`peppers.jpg` & `strawberries.jpg`):** * In **`peppers.jpg`**, the **Hue (H)** channel effectively maps different peppers to distinct gray levels (e.g., the yellow pepper is a different shade than the red one), regardless of how shiny they are. 
    * The **Value (V)** channel captures the "glare" and specular reflections on the skin of the peppers and strawberries. 
    * In **`yellowlily.jpg`**, the **Saturation (S)** channel is very bright on the flower petals but dark on the soil, showing that the flower has high color purity compared to the neutral background.
* **Engineering Significance:** HSV is superior for **color segmentation**. Because Hue is decoupled from brightness (Value), we can identify the "redness" of a strawberry even if it is partially in shadow.

---

#### 1.2 RGB to YCbCr Conversion
* **Observations (`lighthouse.jpg` & `peppers.jpg`):** * The **Luminance (Y)** channel in **`lighthouse.jpg`** looks like a high-quality black-and-white photo, containing all the sharp edges of the building and the textures of the clouds. 
    * The **Cb (Blue-difference)** and **Cr (Red-difference)** channels appear much "flatter" and blurrier. In **`peppers.jpg`**, the red peppers are almost white in the **Cr** channel, while the green ones are dark.
* **Engineering Significance:** This space is the foundation of **image compression (JPEG)**. Since the human visual system is more sensitive to structure (Y) than to color detail (Cb/Cr), engineers can compress the chroma channels more aggressively without a perceptible drop in visual quality.

---

#### 1.3 RGB to L\*a\*b\* Conversion
* **Observations (`nenufares.bmp` & `yellowlily.jpg`):**
    * In **`nenufares.bmp`**, the **a\*** channel (Green-Red axis) creates a massive contrast between the pink flowers (high/white values) and the green lily pads (low/dark values). 
    * In **`yellowlily.jpg`**, the **b\*** channel (Blue-Yellow axis) highlights the yellow petals against the dark soil much more effectively than the standard RGB channels.
* **Engineering Significance:** L\*a\*b\* is designed for **perceptual uniformity**. A Euclidean distance calculated between two colors in L\*a\*b\* space correlates directly to the degree of visual difference perceived by a human, making it the industry standard for color accuracy and measuring color distortion.

---

#### 1.4 Histograms: RGB vs. L\*a\*b\*
* **Observations (`testRGB.bmp` & `peppers.jpg`):**
    * In the **`testRGB.bmp`** gradient, the RGB histogram shows three overlapping, broad distributions. If the brightness changes, all three curves shift simultaneously, making it hard to mathematically define a specific color.
    * In the **L\*a\*b\* histogram** for **`peppers.jpg`**, the **L (Lightness)** distribution describes the overall exposure of the scene, while the **a** and **b** histograms show distinct "spikes" or "clusters" that represent the specific pigments (red, yellow, green) present in the image.
* **Engineering Significance:** L\*a\*b\* histograms allow for **intensity-independent color analysis**. We can adjust the contrast of an image (by stretching the L histogram) without causing a "color cast" or shifting the actual hues of the objects.